# 파일 관리

- 웹사이트를 꾸며줄 CSS나 이미지 파일 등을 처리하는 방법

## 포스트 목록 페이지에 부트스트랩 적용

### HTML에서 css 파일을 불러와 적용하는 과정

1. 서버접속(127.0.0.1:8000/blog/ 접속)

2. 서버에서 클라이언트에 post_list.html 전송

3. 클라이언트로부터 bootstrap.min.css 파일 요청 받음

4. 클라이언트에 bootstrap.min.css 전송

### MTV 구조

- 각 앱의 templates 폴더의 html 파일은 정적인 파일이 아님
    - html 파일에는 빈칸들이 있고 views.py 에 정의한 내용에 따라 빈칸을 채워 사용자에게 정보를 제공
 
- css, js 파일은 html 파일과 다르게 고정된 내용만 제공하면 됨
    - 따라서 웹 서버에서 특정 URL 로 접근하면 해당 css, js 파일을 제공할 수 있도록 설정

#### 1. static 폴더 생성 후 css파일 넣기

#### 2. css 파일 경로 지정

- 템플릿에서 static 파일을 사용하겠다고 선언
    - 부트스트랩 파일 링크 수정

In [ ]:
# blog/post_list.html

{% load static %}
<!DOCTYPE html>
<html>
    <head>
        <title>Blog</title>
        <link href="{% static 'blog/bootstrap/bootstrap.min.css' %}" rel="stylesheet" type="text/css">
        <script src="https://kit.fontawesome.com/30a7c89379.js" crossorigin="anonymous"></script>
    </head>

### 실제 포스트 내용이 표시되도록 수정

In [ ]:
# blog/post_list.html

                    <h1>Blog</h1>
                    {% for p in post_list %}
                        <!-- Blog post-->
                        <div class="card mb-4">
                            <a href="#!"><img class="card-img-top" src="https://dummyimage.com/700x350/dee2e6/6c757d.jpg" alt="..." /></a>
                            <div class="card-body">
                                <div class="small text-muted">{{ p.created_at }}</div>
                                <h2 class="card-title h4">{{ p.title }}</h2>
                                <p class="card-text">{{ p.content }}</p>
                                <a class="btn btn-primary" href="{{ p.get_absolute_url }}">Read more →</a>
                            </div>
                        </div>
                    {% endfor %}
                    <!-- Pagination-->

## 게시글 상세 페이지에 부트스트랩 적용

- startbootstrap > blog post 다운로드

- 실제 포스트 내용이 표시되도록 수정

## 포스트에 이미지 올리기

- 해당 포스트와 관련 있는 이미지를 작성자가 직접 업로드 할 수 있도록 설정

### 이미지 폴더 지정

- 장고에서는 이미지 업로드를 위한 ImageField를 제공함
    - ImageField 를 사용하기 위해서는 사용자가 업로드한 이미지를 어디에 저장할지 먼저 설정해야 함
    - 그리고 업로드된 이미지들이 모여 있는 폴더의 URL을 어떻게 할지도 설정해야 함

In [ ]:
# django_prj/settings.py

MEDIA_URL = "/media/"
MEDIA_ROOT = BASE_DIR / "_media"

- 이미지 파일은 프로젝트 폴더 아래에 "_media" 라는 이름의 폴더를 만들고 그 안에 저장하도록 설정됨

- MEDIA_URL 은 /media/ 로 지정하여 웹브라우저에서 도메인/media/ 라는 경로로 접속하면 미디어 파일을 사용하겠다는 의미

In [ ]:
# blog/models.py

class Post(models.Model):
    title = models.CharField(max_length = 30)
    content = models.TextField()
    head_image = models.ImageField(upload_to="blog/images/%Y/%m/%d/",
                                   blank = True)
    created_at = models.DateTimeField(auto_now_add = True)
    updated_at = models.DateTimeField(auto_now = True)

    def __str__(self):
        return f"[{self.pk}]{self.title}"

- upload_to : 이미지를 저장할 폴더의 경로 규칙을 지정
    - blog 폴더 아래에 images 폴더 아래에 연도 폴더, 월 폴더, 일 폴더까지 내려간 위치에 저장하도록 설정됨
        - 한 폴더에 저장하지 않고 여러 폴더로 내려가서 저장하면 방문자가 웹사이트를 방문했을 때 서버에서 파일을 찾아오는 시간을 단축할 수 있음
     
        - 한 폴더에 너무 많은 파일이 있으면 파일을 찾는 시간이 걸리지만, 폴더를 여러 번 타고 들어가는 것은 시간에 큰 영향을 주지 않음
     
- blank = True
    - 해당 필드는 필수 항목이 아님
    - 필드의 내용을 채우지 않더라도 잘 저장됨

### 미디어 파일을 위한 URL 지정

In [ ]:
# django_prj/urls.py

from django.conf import settings
from django.conf.urls.static import static

urlpatterns = [
    path("admin/", admin.site.urls),
    path("blog/", include("blog.urls")),
    path("", include("single_pages.urls")),
]

urlpatterns += static(settings.MEDIA_URL,
                      document_root = settings.MEDIA_ROOT)

## 포스트에 파일 올리기

- post 모델에 이미지뿐만아니라 다른 종류의 파일도 업로드하고 추후에 방문자가 다운로드 받을 수 있도록 설정

### file_upload 필드 생성

- 모델에 FileField 로 필드를 추가
    - 사용법은 ImageField와 거의 동일

In [ ]:
# blog/models.py

class Post(models.Model):
    title = models.CharField(max_length = 30)
    content = models.TextField()
    head_image = models.ImageField(upload_to = "blog/images/%Y/%m/%d/",
                                   blank = True)
    file_upload = models.FileField(upload_to = "blog/files/%Y/%m/%d/",
                                   blank = True)
    created_at = models.DateTimeField(auto_now_add = True)
    updated_at = models.DateTimeField(auto_now = True)